# Week 3 — Baseline RAG (유방암 정보 검색 RAG)

## 목표
수집한 PDF로 가장 단순한 Naive RAG 파이프라인을 만들고, 5개 이상 질문으로 RAGAS 평가(Faithfulness / Answer Relevancy / Context Precision)를 측정한다. 완벽한 성능보다 **baseline 기록**이 목적.

## 구성
| 항목 | 선택 |
|---|---|
| PDF Parser | PyMuPDF |
| Chunking | RecursiveCharacterTextSplitter (512 / 100) |
| Embedding | `intfloat/multilingual-e5-base` |
| Vector DB | Chroma (로컬) |
| LLM | Ollama `qwen3:4b` |
| 평가 | RAGAS (Faithfulness / Answer Relevancy / Context Precision) |

## 사전 준비
1. Ollama 설치 후 모델 다운로드
   ```bash
   # https://ollama.com/download 에서 설치 후
   ollama pull qwen3:4b
   ollama serve   # 백그라운드로 실행되어 있어야 함
   ```
2. 패키지 설치 (다음 셀)
3. `data/raw/pdf/` 아래에 PDF가 받아져 있어야 함 (week 0 다운로드 스크립트 결과)

---
## 0. 패키지 설치

In [1]:
# 한 번만 실행
%pip install -q langchain langchain-community langchain-chroma langchain-huggingface langchain-ollama
%pip install -q sentence-transformers pymupdf chromadb
%pip install -q ragas datasets pandas tiktoken

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: c:\Users\hfdt\.pyenv\pyenv-win\versions\3.11.9\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: c:\Users\hfdt\.pyenv\pyenv-win\versions\3.11.9\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: c:\Users\hfdt\.pyenv\pyenv-win\versions\3.11.9\python.exe -m pip install --upgrade pip


---
## 1. 경로 / 설정

In [2]:
from pathlib import Path
import os, json

# 프로젝트 루트 기준 (notebook이 notebooks/ 안에 있다고 가정)
PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_EVAL = PROJECT_ROOT / "data" / "eval"
VECTOR_DIR = PROJECT_ROOT / "data" / "vector_store" / "week3_chroma"

for p in [DATA_PROCESSED, DATA_EVAL, VECTOR_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Hyperparams
CHUNK_SIZE = 512
CHUNK_OVERLAP = 100
EMBEDDING_MODEL = "intfloat/multilingual-e5-base"
OLLAMA_MODEL = "qwen3:4b"
TOP_K = 5
COLLECTION_NAME = "breast_rag_week3"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PDF 폴더 확인:")
for src in (DATA_RAW / "pdf").iterdir() if (DATA_RAW / "pdf").exists() else []:
    pdfs = list(src.glob("*.pdf"))
    print(f"  {src.name}: {len(pdfs)}개")

PROJECT_ROOT: d:\rag_agent_project\rag-agent-portfolio
PDF 폴더 확인:
  esmo: 1개
  kbcs: 1개
  ncc: 3개
  nccn: 5개


---
## 2. PDF 로드 (PyMuPDF)

`manifest.json`이 있으면 메타데이터를 chunk에 그대로 붙인다. 없으면 폴더명을 출처로 사용.

In [3]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.documents import Document

# manifest 로드 (다운로드 스크립트가 만들어둔 파일)
manifest_path = DATA_RAW / "metadata" / "manifest.json"
if manifest_path.exists():
    with open(manifest_path, encoding="utf-8") as f:
        manifest = json.load(f)
    # filename -> metadata 매핑
    meta_lookup = {m["filename"]: m for m in manifest if m.get("downloaded")}
    print(f"manifest 로드 완료: {len(meta_lookup)}개 항목")
else:
    meta_lookup = {}
    print("manifest.json 없음 — 폴더명만 메타데이터로 사용")

def load_all_pdfs(pdf_root: Path) -> list[Document]:
    """data/raw/pdf/**/*.pdf 전부 로드하고 메타데이터 부착."""
    all_docs = []
    for pdf_path in pdf_root.rglob("*.pdf"):
        try:
            loader = PyMuPDFLoader(str(pdf_path))
            docs = loader.load()
            extra = meta_lookup.get(pdf_path.name, {})
            for d in docs:
                d.metadata.update({
                    "filename": pdf_path.name,
                    "source_folder": pdf_path.parent.name,
                    "org": extra.get("org", pdf_path.parent.name),
                    "title": extra.get("title", pdf_path.stem),
                    "language": extra.get("language", "unknown"),
                    "doc_type": extra.get("doc_type", "unknown"),
                    "priority": extra.get("priority", "unknown"),
                })
            all_docs.extend(docs)
            print(f"  OK   {pdf_path.name}: {len(docs)} pages")
        except Exception as e:
            print(f"  FAIL {pdf_path.name}: {e}")
    return all_docs

documents = load_all_pdfs(DATA_RAW / "pdf")
print(f"\n총 페이지 수: {len(documents)}")

c:\Users\hfdt\.pyenv\pyenv-win\versions\3.11.9\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


manifest 로드 완료: 13개 항목
  OK   esmo_breast_cancer_patient_guide_korean.pdf: 70 pages
  OK   kbcs_korean_breast_cancer_guideline_2023.pdf: 270 pages
  OK   ncc_breast_cancer_hormone_therapy.pdf: 2 pages
  OK   ncc_breast_cancer_prevention_guide.pdf: 10 pages
  OK   ncc_breast_cancer_screening_guideline_2015.pdf: 114 pages
  OK   nccn_breast_cancer_screening_diagnosis_patient.pdf: 52 pages
  OK   nccn_dcis_patient.pdf: 52 pages
  OK   nccn_inflammatory_breast_cancer_patient.pdf: 74 pages
  OK   nccn_invasive_breast_cancer_patient.pdf: 86 pages
  OK   nccn_metastatic_breast_cancer_patient.pdf: 66 pages

총 페이지 수: 796


### 데이터 sanity check

첫 페이지 1개만 출력해서 PDF가 제대로 파싱됐는지 확인.

In [4]:
if documents:
    print("--- 첫 페이지 메타데이터 ---")
    print(json.dumps(documents[0].metadata, ensure_ascii=False, indent=2))
    print("\n--- 첫 페이지 본문 (앞 500자) ---")
    print(documents[0].page_content[:500])

--- 첫 페이지 메타데이터 ---
{
  "producer": "Adobe PDF Library 15.0",
  "creator": "Adobe InDesign 16.1 (Macintosh)",
  "creationdate": "2021-04-22T10:54:24+01:00",
  "source": "d:\\rag_agent_project\\rag-agent-portfolio\\data\\raw\\pdf\\esmo\\esmo_breast_cancer_patient_guide_korean.pdf",
  "file_path": "d:\\rag_agent_project\\rag-agent-portfolio\\data\\raw\\pdf\\esmo\\esmo_breast_cancer_patient_guide_korean.pdf",
  "total_pages": 70,
  "format": "PDF 1.6",
  "title": "ESMO Breast Cancer Guide for Patients (Korean)",
  "author": "European Society for Medical Oncology",
  "subject": "ESMO",
  "keywords": "European Society for Medical Oncology, ESMO, KO, Breast, Cancer, Guide, Patients, Korean",
  "moddate": "2021-05-01T18:37:20+02:00",
  "trapped": "",
  "modDate": "D:20210501183720+02'00'",
  "creationDate": "D:20210422105424+01'00'",
  "page": 0,
  "filename": "esmo_breast_cancer_patient_guide_korean.pdf",
  "source_folder": "esmo",
  "org": "ESMO",
  "language": "ko",
  "doc_type": "patient_

---
## 3. Chunking (RecursiveCharacterTextSplitter)

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
    length_function=len,
)

chunks = splitter.split_documents(documents)
print(f"전체 chunk 수: {len(chunks)}")
print(f"평균 chunk 길이(문자): {sum(len(c.page_content) for c in chunks) / max(len(chunks),1):.0f}")
print("\n--- chunk 예시 ---")
print(chunks[0].metadata)
print(chunks[0].page_content[:300])

전체 chunk 수: 3214
평균 chunk 길이(문자): 436

--- chunk 예시 ---
{'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign 16.1 (Macintosh)', 'creationdate': '2021-04-22T10:54:24+01:00', 'source': 'd:\\rag_agent_project\\rag-agent-portfolio\\data\\raw\\pdf\\esmo\\esmo_breast_cancer_patient_guide_korean.pdf', 'file_path': 'd:\\rag_agent_project\\rag-agent-portfolio\\data\\raw\\pdf\\esmo\\esmo_breast_cancer_patient_guide_korean.pdf', 'total_pages': 70, 'format': 'PDF 1.6', 'title': 'ESMO Breast Cancer Guide for Patients (Korean)', 'author': 'European Society for Medical Oncology', 'subject': 'ESMO', 'keywords': 'European Society for Medical Oncology, ESMO, KO, Breast, Cancer, Guide, Patients, Korean', 'moddate': '2021-05-01T18:37:20+02:00', 'trapped': '', 'modDate': "D:20210501183720+02'00'", 'creationDate': "D:20210422105424+01'00'", 'page': 0, 'filename': 'esmo_breast_cancer_patient_guide_korean.pdf', 'source_folder': 'esmo', 'org': 'ESMO', 'language': 'ko', 'doc_type': 'patient_guideli

---
## 4. Embedding + Chroma 인덱싱

처음 실행 시 multilingual-e5-base 모델 다운로드(~500MB)에 시간이 좀 걸린다.  
e5 계열은 `passage: ...`, `query: ...` prefix 규칙이 있는데, 일단 baseline에서는 default로 두고 4주차 이후에 비교 실험.

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},   # GPU 있으면 "cuda"
    encode_kwargs={"normalize_embeddings": True},
)
print("embedding 모델 로드 완료")

c:\Users\hfdt\.pyenv\pyenv-win\versions\3.11.9\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hfdt\.cache\huggingface\hub\models--intfloat--multilingual-e5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back

embedding 모델 로드 완료


In [7]:
# 기존 인덱스가 있으면 그대로 사용, 없으면 새로 생성
import chromadb

client = chromadb.PersistentClient(path=str(VECTOR_DIR))
existing = [c.name for c in client.list_collections()]
print("기존 컬렉션:", existing)

if COLLECTION_NAME in existing:
    vectorstore = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embeddings,
        persist_directory=str(VECTOR_DIR),
    )
    print(f"기존 컬렉션 로드: {vectorstore._collection.count()}개 chunk")
else:
    print(f"새 컬렉션 생성 중... ({len(chunks)}개 chunk 인덱싱)")
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=COLLECTION_NAME,
        persist_directory=str(VECTOR_DIR),
    )
    print(f"인덱싱 완료: {vectorstore._collection.count()}개")

기존 컬렉션: []
새 컬렉션 생성 중... (3214개 chunk 인덱싱)


KeyboardInterrupt: 

### 검색 sanity check

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})

test_q = "유방암 검진은 몇 살부터 받아야 하나요?"
hits = retriever.invoke(test_q)
print(f"질문: {test_q}")
print(f"검색 결과 {len(hits)}개\n")
for i, h in enumerate(hits, 1):
    print(f"[{i}] {h.metadata.get('org')} / p.{h.metadata.get('page','?')} / {h.metadata.get('filename')}")
    print("   ", h.page_content[:150].replace("\n", " "), "...\n")

---
## 5. LLM 답변 생성 (Ollama qwen3:4b)

RAG 프롬프트 설계 포인트:
- context 안의 내용만 사용
- 모르면 "문서에서 확인할 수 없습니다"
- 출처(기관 + 파일명) 명시
- 의료 정보 면책 문구 포함

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model=OLLAMA_MODEL, temperature=0.0)

RAG_PROMPT = ChatPromptTemplate.from_template("""당신은 유방암 정보 검색 보조 시스템입니다.

아래 [참고 문서]만 사용해서 [질문]에 답변하세요. 문서에 없는 내용은 추측하지 말고 "제공된 문서에서 확인할 수 없습니다"라고 답하세요.
답변 마지막에는 반드시 다음 두 가지를 포함하세요:
1. 출처: 참고한 문서명과 페이지 (예: 출처: 국립암센터 유방암 검진 권고안, p.5)
2. 면책 문구: "이 답변은 일반 정보 제공 목적이며, 실제 진단·치료는 반드시 의료진과 상의하세요."

[참고 문서]
{context}

[질문]
{question}

[답변]""")

def format_context(docs):
    parts = []
    for i, d in enumerate(docs, 1):
        meta = d.metadata
        header = f"[{i}] 출처: {meta.get('org','?')} / {meta.get('title','?')} / p.{meta.get('page','?')}"
        parts.append(f"{header}\n{d.page_content}")
    return "\n\n---\n\n".join(parts)

def ask(question: str):
    docs = retriever.invoke(question)
    prompt = RAG_PROMPT.format(context=format_context(docs), question=question)
    answer = (llm | StrOutputParser()).invoke(prompt)
    return {"question": question, "answer": answer, "contexts": [d.page_content for d in docs], "retrieved_docs": docs}

result = ask(test_q)
print(result["answer"])

---
## 6. 평가용 질문 세트 (Golden Set v0)

5~10개 질문을 도메인 기준으로 작성. 4주차에서 확장할 거니까 일단 baseline 측정용.

- `ground_truth`는 RAGAS 일부 지표(Context Recall 등)에 쓰이는데, 3주차에서는 Faithfulness / Answer Relevancy / Context Precision만 측정할 거라 비워둬도 됨. 단, 4주차 이후를 위해 가능한 만큼 채워두면 좋다.

In [ ]:
import pandas as pd

golden = [
    {"question": "유방암 검진은 몇 살부터 받는 것이 권장되나요?",
     "ground_truth": "국립암센터 권고안은 만 40세 이상 여성에게 2년마다 유방촬영술 검진을 권고합니다."},
    {"question": "유방암의 주요 위험 요인은 무엇인가요?",
     "ground_truth": "고연령, 가족력, BRCA1/2 등 유전적 요인, 이른 초경/늦은 폐경, 호르몬 노출, 비만 등이 알려져 있습니다."},
    {"question": "HER2 양성 유방암이란 무엇인가요?",
     "ground_truth": "HER2 단백이 과발현된 유방암으로, 표적치료제(트라스투주맙 등)가 효과적입니다."},
    {"question": "유방암 1기와 2기의 차이는 무엇인가요?",
     "ground_truth": "종양 크기와 림프절 전이 여부에 따라 병기가 나뉘며, 1기는 종양이 작고 림프절 전이 없음, 2기는 종양이 크거나 일부 림프절 전이가 있는 상태입니다."},
    {"question": "DCIS(상피내암)는 침윤성 유방암과 어떻게 다른가요?",
     "ground_truth": "DCIS는 유관 내에 머무르는 0기 암이며, 침윤성 유방암은 기저막을 뚫고 주변 조직으로 침범한 상태입니다."},
    {"question": "항호르몬제 치료는 어떤 환자에게 사용되나요?",
     "ground_truth": "호르몬 수용체 양성(ER+/PR+) 유방암 환자에게 사용되며, 타목시펜이나 아로마타제 억제제가 대표적입니다."},
    {"question": "유방절제술 후 재건수술은 어떤 옵션이 있나요?",
     "ground_truth": "보형물 재건, 자가조직 재건(복부/등 피판 등), 또는 두 방식의 조합이 있습니다."},
    {"question": "BRCA 유전자 검사는 누구에게 권장되나요?",
     "ground_truth": "가족 중 유방암·난소암 환자가 있거나, 젊은 나이에 발병한 경우 등 유전성 위험이 의심될 때 권장됩니다."},
    {"question": "전이성 유방암(4기)의 일반적인 치료 목표는 무엇인가요?",
     "ground_truth": "완치보다는 증상 완화와 생존 연장, 삶의 질 유지가 주된 목표입니다."},
    {"question": "유방암 환자가 식이요법에서 주의할 점은?",
     "ground_truth": "균형 잡힌 식단, 적정 체중 유지, 음주 제한이 권장되며 특정 식품의 효과는 근거가 제한적입니다."},
]

df_golden = pd.DataFrame(golden)
golden_path = DATA_EVAL / "golden_set_v0.csv"
df_golden.to_csv(golden_path, index=False, encoding="utf-8-sig")
print(f"저장: {golden_path}")
df_golden

---
## 7. 전체 질문에 대해 RAG 실행 → 결과 수집

qwen3:4b로 10개 답변 생성하는데 시간이 좀 걸린다. CPU 기준 10~20분 예상.

In [ ]:
from tqdm import tqdm

results = []
for q in tqdm(df_golden["question"].tolist(), desc="RAG"):
    r = ask(q)
    results.append({
        "question": r["question"],
        "answer": r["answer"],
        "contexts": r["contexts"],
    })

# ground_truth 결합
for r, g in zip(results, golden):
    r["ground_truth"] = g["ground_truth"]

df_results = pd.DataFrame(results)
df_results.to_json(DATA_PROCESSED / "week3_rag_results.json", orient="records", force_ascii=False, indent=2)
print("저장 완료")
df_results[["question", "answer"]]

---
## 8. RAGAS 평가

RAGAS는 내부적으로 LLM을 호출해 평가한다. qwen3:4b로 평가하면 느리므로, 평가 LLM은 별도로 지정 가능.

**중요**: RAGAS도 generation/critic LLM이 필요. 일단 동일하게 ollama 사용. 너무 느리거나 점수 산정 실패 시 아래 옵션 활용:
- (Option A) ollama 더 큰 모델로 평가: `qwen3:8b`
- (Option B) Anthropic Claude API로 평가: `ChatAnthropic` 사용 (API key 필요)

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# RAGAS용 LLM/Embedding 래핑
ragas_llm = LangchainLLMWrapper(ChatOllama(model=OLLAMA_MODEL, temperature=0.0))
ragas_emb = LangchainEmbeddingsWrapper(embeddings)

# RAGAS는 contexts 필드를 list[str]로 요구
ragas_dataset = Dataset.from_list([
    {
        "question": r["question"],
        "answer": r["answer"],
        "contexts": r["contexts"],
        "ground_truth": r["ground_truth"],
    }
    for r in results
])

scores = evaluate(
    dataset=ragas_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision],
    llm=ragas_llm,
    embeddings=ragas_emb,
)
print(scores)

In [ ]:
# 결과를 DataFrame으로 정리해 저장
df_scores = scores.to_pandas()
df_scores.to_csv(DATA_PROCESSED / "week3_ragas_scores.csv", index=False, encoding="utf-8-sig")

print("--- Baseline 평균 점수 ---")
for col in ["faithfulness", "answer_relevancy", "context_precision"]:
    if col in df_scores.columns:
        print(f"  {col}: {df_scores[col].mean():.4f}")

df_scores

---
## 9. 회고용 — 잘못 검색된 chunk 찾기

4주차 retrospective에서 "잘못 검색된 chunk 예시 2개 이상"이 필요하다.  
Context Precision이 낮은 질문 위주로 확인하면 좋다.

In [ ]:
# context_precision이 낮은 질문 top 3 출력 → 검색 결과 직접 확인
if "context_precision" in df_scores.columns:
    worst = df_scores.nsmallest(3, "context_precision")
    for idx, row in worst.iterrows():
        print("=" * 60)
        print(f"질문: {row['question']}")
        print(f"context_precision: {row['context_precision']:.3f}")
        print(f"\n답변: {row['answer'][:300]}...")
        print("\n검색된 chunk:")
        for i, c in enumerate(row["contexts"], 1):
            print(f"  [{i}] {c[:200].replace(chr(10), ' ')}...")
        print()

---
## 10. 메모 (3주차 회고용)

`docs/week3_baseline_rag.md` 또는 `docs/week4_retrospective.md`에 옮겨 적기 위한 메모.

- **데이터**: PDF 파일 N개, 총 chunk 수 ?, 평균 길이 ?
- **chunking 설정**: 512 / 100 (Recursive)
- **embedding**: multilingual-e5-base
- **LLM**: qwen3:4b (Ollama)
- **RAGAS baseline**:
  - Faithfulness: ?
  - Answer Relevancy: ?
  - Context Precision: ?
- **잘 된 점**:
- **아쉬운 점**:
- **잘못 검색된 chunk 예시 2개**:
  - (예시 1) 질문 / chunk / 왜 잘못됐는지
  - (예시 2) ...
- **4주차에서 시도할 것**:
  - chunk_size 변경 비교
  - 도메인 특화 splitter (페이지/섹션 기반)
  - 메타데이터 필드 확장 활용